<a href="https://colab.research.google.com/github/bill105/llm-bill/blob/main/1234test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installation de la bibliothèque de lecture
!pip install -q tokenizers

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. On crée le modèle BPE (la même technologie de lecture que GPT-4 et Claude)
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

# 2. On définit la taille du dictionnaire (5000 mots/symboles, parfait pour commencer)
trainer = BpeTrainer(
    vocab_size=5000,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

# 3. On crée un fichier texte contenant du français et du code (le futur du LLM)
donnees_entrainement = [
    "def additionner(a, b):\n    return a + b\n",
    "Créez une fonction pour calculer la vitesse.",
    "variable = int(input('Entrez un nombre'))",
    "Le samouraï frappe le bois avec précision.",
    "for i in range(10):\n    print('Bonjour le monde')",
    "Si la condition est vraie, exécute la boucle.",
    "class Guerrier:\n    def __init__(self):\n        self.vie = 100",
    "L'intelligence artificielle apprend de ses erreurs."
]

with open("dojo_francais_code.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(donnees_entrainement))

# 4. On entraîne le tokéniseur sur ce fichier
print("🏋️ Entraînement du Tokéniseur (Les Yeux) sur le français et le code...")
tokenizer.train(files=["dojo_francais_code.txt"], trainer=trainer)
tokenizer.save("tokenizer_samourai.json")
print("✅ Tokéniseur sauvegardé avec succès !")

# TEST : Regardons comment votre IA va lire le français et le code
phrase_test = "def samourai(): return 'victoire'"
encodage = tokenizer.encode(phrase_test)
print(f"\nPhrase de test : {phrase_test}")
print(f"Mots découpés (Tokens) : {encodage.tokens}")
print(f"Traduits en chiffres (IDs) : {encodage.ids}")

🏋️ Entraînement du Tokéniseur (Les Yeux) sur le français et le code...
✅ Tokéniseur sauvegardé avec succès !

Phrase de test : def samourai(): return 'victoire'
Mots découpés (Tokens) : ['def', 'sam', 'our', 'a', 'i', '(', '):', 'return', "'", 'v', 'i', 'ct', 'o', 'i', 're', "'"]
Traduits en chiffres (IDs) : [75, 126, 57, 21, 28, 5, 55, 190, 4, 39, 28, 93, 33, 28, 47, 4]


In [ ]:
import torch
import torch.nn as nn

# ==========================================
# 1. LE SENSEUR DU MILIEU (L'instinct de survie)
# ==========================================
class SenseurEnvironnement:
    @staticmethod
    def verifier_ressources():
        """Le samouraï ressent la VRAM du GPU T4 de Colab"""
        if torch.cuda.is_available():
            vram_libre, vram_total = torch.cuda.mem_get_info()
            vram_libre_go = vram_libre / 1e9
            return vram_libre_go
        return 8.0 # Simule 8 Go si pas de GPU

# ==========================================
# 2. L'ARCHITECTURE DU CERVEAU (De zéro)
# ==========================================
class SamouraiBlock(nn.Module):
    def __init__(self, dim_modele, nombre_tetes, taux_dropout=0.1):
        super().__init__()
        self.attention = nn.MultiheadAttention(dim_modele, nombre_tetes, batch_first=True, dropout=taux_dropout)
        self.reseau_pensee = nn.Sequential(
            nn.Linear(dim_modele, dim_modele * 4),
            nn.GELU(),
            nn.Dropout(taux_dropout),
            nn.Linear(dim_modele * 4, dim_modele)
        )
        self.norme1 = nn.LayerNorm(dim_modele)
        self.norme2 = nn.LayerNorm(dim_modele)

    def forward(self, x):
        taille = x.size(1)
        # Le samouraï ne voit pas l'avenir, il se base sur le passé (masque causal)
        masque = torch.triu(torch.ones(taille, taille, device=x.device), diagonal=1).bool()
        attention_out, _ = self.attention(x, x, x, attn_mask=masque)
        x = self.norme1(x + attention_out)
        pensee_out = self.reseau_pensee(x)
        x = self.norme2(x + pensee_out)
        return x

class MonLLMSamourai(nn.Module):
    def __init__(self, taille_vocabulaire, dim_modele=384, nombre_blocs=6, nombre_tetes=6, taille_contexte=256):
        super().__init__()
        print("🥷 Naissance du Samouraï LLM (Conscient de son milieu)...")
        self.taille_contexte = taille_contexte
        self.embedding_mot = nn.Embedding(taille_vocabulaire, dim_modele)
        self.embedding_position = nn.Embedding(taille_contexte, dim_modele)

        # Le cerveau : 6 blocs de réflexion
        self.blocs_samourai = nn.ModuleList([
            SamouraiBlock(dim_modele, nombre_tetes) for _ in range(nombre_blocs)
        ])

        self.norme_finale = nn.LayerNorm(dim_modele)
        self.sortie = nn.Linear(dim_modele, taille_vocabulaire)
        self._init_poids()

    def _init_poids(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)

    def forward(self, x):
        if x.size(1) > self.taille_contexte:
            x = x[:, -self.taille_contexte:]

        positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
        x = self.embedding_mot(x) + self.embedding_position(positions)

        # ⚔️ LE BUSHIDO ADAPTATIF : L'IA ressent son environnement
        vram_libre = SenseurEnvironnement.verifier_ressources()

        if vram_libre < 2.0:
            # MODE SURVIE : On ne fait passer que le premier bloc
            print(f"🌿 Environnement faible ({vram_libre:.2f} Go libres). Mode survie activé.")
            x = self.blocs_samourai[0](x)
        else:
            # MODE COMBAT : Pleine puissance
            for bloc in self.blocs_samourai:
                x = bloc(x)

        x = self.norme_finale(x)
        return self.sortie(x)

# ==========================================
# TEST DE L'ÉTAPE 2
# ==========================================
print("Chargement du cerveau sur le GPU T4...")
device = "cuda" if torch.cuda.is_available() else "cpu"

# On utilise le vocabulaire de 5000 mots du tokéniseur
TAILLE_VOCAB = 5000
samourai = MonLLMSamourai(taille_vocabulaire=TAILLE_VOCAB).to(device)

# On compte ses muscles (paramètres)
nb_params = sum(p.numel() for p in samourai.parameters())
print(f"🧠 Le Samouraï possède {nb_params / 1e6:.2f} Millions de paramètres.")

# On vérifie la VRAM du T4 au démarrage
vram_libre = SenseurEnvironnement.verifier_ressources()
print(f"🗡️ VRAM disponible sur le T4 : {vram_libre:.2f} Go")

# On simule une phrase de 10 mots au hasard pour voir si le cerveau réagit
phrase_test = torch.randint(0, TAILLE_VOCAB, (1, 10)).to(device)
prediction = samourai(phrase_test)

print(f"✅ Forme de la pensée : {prediction.shape} (Le cerveau a prédit 10 mots de sortie)")

Chargement du cerveau sur le GPU T4...
🥷 Naissance du Samouraï LLM (Conscient de son milieu)...
🧠 Le Samouraï possède 14.59 Millions de paramètres.
🗡️ VRAM disponible sur le T4 : 15.46 Go
✅ Forme de la pensée : torch.Size([1, 10, 5000]) (Le cerveau a prédit 10 mots de sortie)


In [ ]:
# ==========================================
# ÉTAPE 3 : LE DOJO (L'Entraînement du Samouraï)
# ==========================================
from torch.optim import AdamW
from tokenizers import Tokenizer

# 1. On charge le Tokéniseur que l'on a créé à l'Étape 1
tokenizer = Tokenizer.from_file("tokenizer_samourai.json")

# On récupère le texte qui contient le français et le code
texte_entier = open("dojo_francais_code.txt", "r", encoding="utf-8").read()

# On transforme tout le texte en chiffres (IDs)
ids = tokenizer.encode(texte_entier).ids

# On met ça sous forme de Tensor pour le GPU T4
data_tensor = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

# 2. L'optimiseur : c'est le "Maître" qui corrige le samouraï à chaque erreur
optimiseur = AdamW(samourai.parameters(), lr=3e-4)

print("🗡️ Début de l'entraînement (Le Samouraï apprend à lire le code et le français)...")

# 3. La boucle d'entraînement (50 tours de combat)
for epoch in range(1, 51):
    # Le samouraï regarde le passé (x) et doit deviner le mot suivant (y)
    x = data_tensor[:, :-1]
    y = data_tensor[:, 1:]

    # Le cerveau réfléchit
    predictions = samourai(x)

    # Le maître calcule l'erreur (Plus c'est proche de 0, plus il est intelligent)
    loss_function = nn.CrossEntropyLoss()
    # On doit adapter la forme des données pour la fonction de perte
    perte = loss_function(predictions.view(-1, TAILLE_VOCAB), y.view(-1))

    # Le samouraï corrige sa posture (rétropropagation)
    optimiseur.zero_grad()
    perte.backward()
    optimiseur.step()

    # On affiche la progression tous les 10 tours
    if epoch % 10 == 0 or epoch == 1:
        print(f"Tour {epoch}/50 - Erreur (Perte) : {perte.item():.4f}")

print("✅ Entraînement terminé ! Le Samouraï s'est familiarisé avec son milieu.")

🗡️ Début de l'entraînement (Le Samouraï apprend à lire le code et le français)...
Tour 1/50 - Erreur (Perte) : 8.5491
Tour 10/50 - Erreur (Perte) : 2.5529
Tour 20/50 - Erreur (Perte) : 0.6927
Tour 30/50 - Erreur (Perte) : 0.2578
Tour 40/50 - Erreur (Perte) : 0.1209
Tour 50/50 - Erreur (Perte) : 0.0701
✅ Entraînement terminé ! Le Samouraï s'est familiarisé avec son milieu.


In [ ]:
# ==========================================
# ÉTAPE 4 : LA VOIX (Le Samouraï génère du code)
# ==========================================
def generer_code(modele, tokenizer, prompt, max_nouveaux_tokens=50):
    print(f"\n🗡️ Le Maître donne l'ordre : '{prompt}'")

    # 1. On convertit le texte de départ en chiffres
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    # 2. La boucle de création
    for _ in range(max_nouveaux_tokens):
        # On coupe si la phrase devient trop longue pour la mémoire du LLM
        if input_ids.size(1) >= 256:
            input_ids = input_ids[:, -255:]

        # Le samouraï réfléchit à partir du texte actuel
        with torch.no_grad():
            logits = modele(input_ids)

        # On regarde la prédiction du TOUT DERNIER mot généré
        dernier_logits = logits[:, -1, :]

        # Le Samouraï choisit le mot (token) avec la plus haute probabilité de logique
        prochain_id = torch.argmax(dernier_logits, dim=-1, keepdim=True)

        # Si le Samouraï décide de s'arrêter (token de fin), on casse la boucle
        if prochain_id.item() == tokenizer.token_to_id("[EOS]"):
            break

        # On ajoute le nouveau mot à la phrase
        input_ids = torch.cat([input_ids, prochain_id], dim=1)

    # 3. On reconvertit les chiffres en vrai texte (français/code)
    texte_genere = tokenizer.decode(input_ids[0].tolist())
    print(f"🥷 Le Samouraï a écrit :\n{texte_genere}")
    return texte_genere

# ==========================================
# TEST DE VOTRE LLM !
# ==========================================
# On lui donne un mot de code, et il doit continuer tout seul
generer_code(samourai, tokenizer, "def", max_nouveaux_tokens=40)
generer_code(samourai, tokenizer, "Le", max_nouveaux_tokens=40)


🗡️ Le Maître donne l'ordre : 'def'
🥷 Le Samouraï a écrit :
def additionner ( a , b ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range

🗡️ Le Maître donne l'ordre : 'Le'
🥷 Le Samouraï a écrit :
Le additionner ( a , b ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range


"Le additionner ( a , b ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range"

In [ ]:
# ==========================================
# LE DOJO COMPLET : De zéro à la méthode AÉCS
# ==========================================
!pip install -q tokenizers

import torch
import torch.nn as nn
from torch.optim import AdamW
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import ast

# 1. Préparation du GPU T4 de Colab
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🗡️ Arme utilisée : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# 2. Les Yeux (Tokéniseur)
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=5000, special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"])

donnees = [
    "def additionner(a, b):\n    return a + b\n",
    "Créez une fonction pour calculer la vitesse.",
    "variable = int(input('Entrez un nombre'))",
    "Le samouraï frappe le bois avec précision.",
    "for i in range(10):\n    print('Bonjour le monde')",
    "class Guerrier:\n    def __init__(self):\n        self.vie = 100",
]
with open("dojo.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(donnees))
tokenizer.train(files=["dojo.txt"], trainer=trainer)

# 3. Le Cerveau (Le LLM avec Senseur Environnemental)
class SenseurEnvironnement:
    @staticmethod
    def verifier_ressources():
        if torch.cuda.is_available(): return torch.cuda.mem_get_info()[0] / 1e9
        return 8.0

class SamouraiBlock(nn.Module):
    def __init__(self, dim, tetes):
        super().__init__()
        self.attention = nn.MultiheadAttention(dim, tetes, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
    def forward(self, x):
        masque = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), diagonal=1).bool()
        x = self.norm1(x + self.attention(x, x, x, attn_mask=masque)[0])
        x = self.norm2(x + self.mlp(x))
        return x

class MonLLMSamourai(nn.Module):
    def __init__(self, vocab, dim=384, blocs=6, tetes=6):
        super().__init__()
        self.emb = nn.Embedding(vocab, dim)
        self.pos = nn.Embedding(256, dim)
        self.blocs = nn.ModuleList([SamouraiBlock(dim, tetes) for _ in range(blocs)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab)
    def forward(self, x):
        if x.size(1) > 256: x = x[:, -255:]
        x = self.emb(x) + self.pos(torch.arange(x.size(1), device=x.device).unsqueeze(0))
        if SenseurEnvironnement.verifier_ressources() < 2.0:
            x = self.blocs[0](x) # Mode survie
        else:
            for b in self.blocs: x = b(x)
        return self.head(self.norm(x))

TAILLE_VOCAB = 5000
samourai = MonLLMSamourai(TAILLE_VOCAB).to(device)
print(f"🥷 Samouraï créé avec {sum(p.numel() for p in samourai.parameters())/1e6:.2f}M paramètres.")

# 4. Les Données
texte = open("dojo.txt", "r", encoding="utf-8").read()
ids = tokenizer.encode(texte).ids
data_tensor = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

# 5. LE BUSHIDO (Votre méthode AÉCS)
def refuteur_aecs(code):
    for flaw in ["sans raison", "magie", "tomber vers le haut"]:
        if flaw in code.lower(): return -10.0
    try:
        ast.parse(code)
        return 2.0 # Le code est logique
    except:
        return -1.0 # Le code est cassé

print("⚔️ Intégration de la méthode AÉCS (Apprentissage Contre-Factuel)...")
optimiseur = AdamW(samourai.parameters(), lr=3e-4)
samourai.train()
loss_function = nn.CrossEntropyLoss()

for epoch in range(1, 51):
    x, y = data_tensor[:, :-1], data_tensor[:, 1:]
    preds = samourai(x)
    perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

    # L'IA est jugée par le réfutateur
    ids_pred = torch.argmax(preds, dim=-1)[0]
    texte_gen = tokenizer.decode(ids_pred.tolist())
    score = refuteur_aecs(texte_gen)

    # La formule de perte AÉCS
    perte_aecs = perte_prob - (0.5 * score)

    optimiseur.zero_grad()
    perte_aecs.backward()
    optimiseur.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Tour {epoch}/50 | Erreur Standard : {perte_prob.item():.4f} | Score Logique AÉCS : {score}")

print("✅ Entraînement terminé ! Le Samouraï a appris par la logique pure.")

🗡️ Arme utilisée : Tesla T4
🥷 Samouraï créé avec 14.59M paramètres.
⚔️ Intégration de la méthode AÉCS (Apprentissage Contre-Factuel)...
Tour 1/50 | Erreur Standard : 8.7161 | Score Logique AÉCS : 2.0
Tour 10/50 | Erreur Standard : 0.3817 | Score Logique AÉCS : -1.0
Tour 20/50 | Erreur Standard : 0.0557 | Score Logique AÉCS : -1.0
Tour 30/50 | Erreur Standard : 0.0302 | Score Logique AÉCS : -1.0
Tour 40/50 | Erreur Standard : 0.0211 | Score Logique AÉCS : -1.0
Tour 50/50 | Erreur Standard : 0.0167 | Score Logique AÉCS : -1.0
✅ Entraînement terminé ! Le Samouraï a appris par la logique pure.


In [ ]:
def parler_au_samourai(prompt, max_tokens=40):
    print(f"\n🗡️ Le Maître ordonne : '{prompt}'")

    # On met l'IA en mode "combat" (sans apprentissage, juste de la réflexion)
    samourai.eval()

    # On traduit le prompt en chiffres
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_tokens):
            # L'IA réfléchit
            logits = samourai(input_ids)
            # Elle choisit le mot le plus logique (Greedy decoding)
            prochain_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)

            # Si elle dit "Fin", on arrête
            if prochain_id.item() == tokenizer.token_to_id("[EOS]"):
                break

            # On ajoute le mot à la phrase
            input_ids = torch.cat([input_ids, prochain_id], dim=1)

    # On reconvertit en texte
    texte_genere = tokenizer.decode(input_ids[0].tolist())
    print(f"🥷 Le Samouraï répond :\n{texte_genere}")
    return texte_genere

# Test 1 : On lui donne un mot de code
parler_au_samourai("def")

# Test 2 : On lui donne un mot en français
parler_au_samourai("Le")


🗡️ Le Maître ordonne : 'def'
🥷 Le Samouraï répond :
def additionner ( a , b ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range

🗡️ Le Maître ordonne : 'Le'
🥷 Le Samouraï répond :
Le additionner ( a , b ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range


"Le additionner ( a , b ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range"

In [ ]:
# 1. On renomme les fichiers pour corriger les fautes de frappe
!mv 'main?.py' model.py
!mv traint.py train.py

# 2. On vérifie que tout est propre
!ls -l

# 3. On lance l'entraînement AÉCS !
!python train.py

total 20
-rw-r--r-- 1 root root 2628 Aug 25 21:13 generate.py
-rw-r--r-- 1 root root 3196 Aug 25 21:13 model.py
-rw-r--r-- 1 root root  157 Aug 25 21:13 README.md
-rw-r--r-- 1 root root 1822 Aug 25 21:13 tokenizer.py
-rw-r--r-- 1 root root 3123 Aug 25 21:13 train.py
🗡️ Arme utilisée : Tesla T4
🏋️ Entraînement du Tokéniseur (Les Yeux) sur le français et le code...
[00:00:00] Tokenize words                 ██████████████████ 49       /       49
[00:00:00] Count pairs                    ██████████████████ 49       /       49
[00:00:00] Compute merges                 ██████████████████ 115      /      115
✅ Tokéniseur sauvegardé sous 'tokenizer_samourai.json' !
🥷 Samouraï créé avec 10.87M paramètres.
⚔️ Intégration de la méthode AÉCS (Apprentissage Contre-Factuel)...
Tour 1/50 | Erreur Standard : 5.1371 | Score Logique AÉCS : -1.0
Tour 10/50 | Erreur Standard : 0.2988 | Score Logique AÉCS : -1.0
Tour 20/50 | Erreur Standard : 0.0623 | Score Logique AÉCS : -1.0
Tour 30/50 | Erreur Standard 

In [ ]:
# On teste le Samouraï
!python generate.py --prompt "def"


🗡️ Le Maître ordonne : 'def'
🥷 Le Samouraï réfléchit...
🏆 Le Samouraï a écrit :
def additionner ( a , b ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range


In [ ]:
import torch
import torch.nn as nn
from torch.optim import AdamW
import ast
from model import MonLLMSamourai
from tokenizers import Tokenizer
from tokenizer import entrainer_et_sauvegarder_tokenizer
import os

# 1. On recrée le tokenizer sur ce nouveau gros fichier
entrainer_et_sauvegarder_tokenizer()
tokenizer = Tokenizer.from_file("tokenizer_samourai.json")

# 2. On charge le gros dataset
device = "cuda" if torch.cuda.is_available() else "cpu"
with open("dataset.txt", "r", encoding="utf-8") as f:
    texte = f.read()
ids = tokenizer.encode(texte).ids
data_tensor = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

# 3. Naissance du cerveau
TAILLE_VOCAB = tokenizer.get_vocab_size()
samourai = MonLLMSamourai(taille_vocabulaire=TAILLE_VOCAB).to(device)

optimiseur = AdamW(samourai.parameters(), lr=3e-4)
loss_function = nn.CrossEntropyLoss()

# La méthode AÉCS (Le Bushido)
def refuteur_aecs(code):
    for flaw in ["sans raison", "magie"]:
        if flaw in code.lower(): return -10.0
    try:
        ast.parse(code); return 2.0
    except: return -1.0

print("⚔️ Début du grand entraînement AÉCS (100 tours sur 1500 exemples)...")
samourai.train()

for epoch in range(1, 101):
    x, y = data_tensor[:, :-1], data_tensor[:, 1:]
    preds = samourai(x)
    perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

    # AÉCS
    ids_pred = torch.argmax(preds, dim=-1)[0]
    texte_gen = tokenizer.decode(ids_pred.tolist())
    score = refuteur_aecs(texte_gen)
    perte_aecs = perte_prob - (0.5 * score)

    optimiseur.zero_grad()
    perte_aecs.backward()
    optimiseur.step()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Tour {epoch}/100 | Erreur : {perte_prob.item():.4f} | Logique : {score}")

# Sauvegarde du nouveau cerveau
torch.save(samourai.state_dict(), "samourai_cerveau.pth")
print("✅ Grand entraînement terminé ! Le cerveau est nettement plus intelligent.")

🏋️ Entraînement du Tokéniseur (Les Yeux) sur le français et le code...
✅ Tokéniseur sauvegardé sous 'tokenizer_samourai.json' !
⚔️ Début du grand entraînement AÉCS (100 tours sur 1500 exemples)...
Tour 1/100 | Erreur : 5.0327 | Logique : -1.0
Tour 10/100 | Erreur : 0.2730 | Logique : -1.0
Tour 20/100 | Erreur : 0.0602 | Logique : -1.0
Tour 30/100 | Erreur : 0.0342 | Logique : -1.0
Tour 40/100 | Erreur : 0.0244 | Logique : -1.0
Tour 50/100 | Erreur : 0.0193 | Logique : -1.0
Tour 60/100 | Erreur : 0.0161 | Logique : -1.0
Tour 70/100 | Erreur : 0.0140 | Logique : -1.0
Tour 80/100 | Erreur : 0.0123 | Logique : -1.0
Tour 90/100 | Erreur : 0.0110 | Logique : -1.0
Tour 100/100 | Erreur : 0.0099 | Logique : -1.0
✅ Grand entraînement terminé ! Le cerveau est nettement plus intelligent.


In [ ]:
!python generate.py --prompt "# Fais une fonction"
!python generate.py --prompt "for i in"


🗡️ Le Maître ordonne : '# Fais une fonction'
🥷 Le Samouraï réfléchit...
🏆 Le Samouraï a écrit :
a is une fonction ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range ( 10 ): print ('

🗡️ Le Maître ordonne : 'for i in'
🥷 Le Samouraï réfléchit...
🏆 Le Samouraï a écrit :
for i in a , b ): return a + b Créez une fonction pour calculer la vitesse . variable = int ( input (' Entrez un nombre ')) Le samouraï frappe le bois avec précision . for i in range ( 10


In [ ]:
import random, ast, torch
import torch.nn as nn
from torch.optim import AdamW
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from model import MonLLMSamourai

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. CRÉATION DU VRAI DATASET (1500 lignes variées)
print("📊 Génération du dataset de 1500 lignes...")
with open("dataset.txt", "w", encoding="utf-8") as f:
    for _ in range(500):
        a, b = random.choice(['a', 'x', 'val']), random.choice(['b', 'y', 'val2'])
        op = random.choice(['+', '-', '*'])
        nom = random.choice(['calculer', 'operation', 'math'])
        f.write(f"# Fais une fonction pour {nom}\n")
        f.write(f"def {nom}({a}, {b}):\n    return {a} {op} {b}\n\n")
    for _ in range(500):
        i = random.choice(['i', 'j', 'k'])
        num = random.randint(2, 50)
        msg = random.choice(['Bonjour', 'Samourai', 'Test', 'Code'])
        f.write(f"# Fais une boucle {num} fois\n")
        f.write(f"for {i} in range({num}):\n    print('{msg}')\n\n")

# 2. ENTRAÎNEMENT DU TOKÉNISEUR
print("👀 Entraînement du Tokéniseur...")
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=5000, special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"])
tokenizer.train(files=["dataset.txt"], trainer=trainer)
tokenizer.save("tokenizer_samourai.json")

# 3. PRÉPARATION DU CERVEAU
TAILLE_VOCAB = tokenizer.get_vocab_size()
samourai = MonLLMSamourai(taille_vocabulaire=TAILLE_VOCAB).to(device)
optimiseur = AdamW(samourai.parameters(), lr=3e-4)
loss_function = nn.CrossEntropyLoss()

texte = open("dataset.txt", "r", encoding="utf-8").read()
data_tensor = torch.tensor(tokenizer.encode(texte).ids, dtype=torch.long, device=device).unsqueeze(0)

# 4. LA MÉTHODE AÉCS
def refuteur_aecs(code):
    for flaw in ["sans raison", "magie"]:
        if flaw in code.lower(): return -10.0
    try:
        ast.parse(code); return 2.0
    except: return -1.0

print("⚔️ Début de l'entraînement AÉCS (100 tours)...")
samourai.train()
TAILLE_CONTEXTE = 256 # La mémoire court-terme de l'IA

for epoch in range(1, 101):
    # LA CORRECTION : On prend un bloc de 256 mots au hasard dans le grand fichier
    start = random.randint(0, data_tensor.size(1) - TAILLE_CONTEXTE - 1)
    x = data_tensor[:, start : start + TAILLE_CONTEXTE]
    y = data_tensor[:, start + 1 : start + TAILLE_CONTEXTE + 1]

    preds = samourai(x)
    perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

    ids_pred = torch.argmax(preds, dim=-1)[0]
    texte_gen = tokenizer.decode(ids_pred.tolist())
    score = refuteur_aecs(texte_gen)
    perte_aecs = perte_prob - (0.5 * score)

    optimiseur.zero_grad()
    perte_aecs.backward()
    optimiseur.step()
    if epoch % 20 == 0:
        print(f"Tour {epoch}/100 | Erreur : {perte_prob.item():.4f} | Logique : {score}")

torch.save(samourai.state_dict(), "samourai_cerveau.pth")
print("✅ Cerveau sauvegardé !")

# 5. TEST DE GÉNÉRATION
def generer(prompt, max_tokens=30):
    print(f"\n🗡️ Le Maître ordonne : '{prompt}'")
    samourai.eval()
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_tokens):
            logits = samourai(input_ids)
            prochain_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, prochain_id], dim=1)
    print(f"🥷 Le Samouraï a écrit :\n{tokenizer.decode(input_ids[0].tolist())}")

generer("# Fais une fonction")
generer("for i in")

📊 Génération du dataset de 1500 lignes...
👀 Entraînement du Tokéniseur...
⚔️ Début de l'entraînement AÉCS (100 tours)...
Tour 20/100 | Erreur : 1.2207 | Logique : 2.0
Tour 40/100 | Erreur : 0.8102 | Logique : -1.0
Tour 60/100 | Erreur : 0.8938 | Logique : -1.0
Tour 80/100 | Erreur : 0.7728 | Logique : -1.0
Tour 100/100 | Erreur : 0.7503 | Logique : -1.0
✅ Cerveau sauvegardé !

🗡️ Le Maître ordonne : '# Fais une fonction'
🥷 Le Samouraï a écrit :
# Fais une fonction pour calculer def calculer ( a , b # Fais une fonction pour calculer def calculer ( a , b # Fais une fonction pour calculer ( a , b

🗡️ Le Maître ordonne : 'for i in'
🥷 Le Samouraï a écrit :
for i in range ( 10 ): print (' Test ') # Fais une boucle 10 fois for k in range ( 10 fois for k in range ( 10 ): print ('


In [ ]:
import torch

def generer_creatif(prompt, max_tokens=50, temperature=0.7):
    """Génère du code avec de la créativité (Temperature)"""
    print(f"\n🗡️ Le Maître ordonne : '{prompt}'")
    samourai.eval()
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_tokens):
            # On coupe la mémoire si trop longue
            if input_ids.size(1) >= 256:
                input_ids = input_ids[:, -255:]

            logits = samourai(input_ids)

            # LA MAGIE : On applique la température
            logits = logits[:, -1, :] / temperature

            # On transforme en probabilités
            probs = torch.softmax(logits, dim=-1)

            # On tire au sort le mot (au lieu de prendre toujours le max)
            prochain_id = torch.multinomial(probs, num_samples=1)

            if prochain_id.item() == tokenizer.token_to_id("[EOS]"):
                break

            input_ids = torch.cat([input_ids, prochain_id], dim=1)

    print(f"🥷 Le Samouraï a inventé :\n{tokenizer.decode(input_ids[0].tolist())}")

# Testez plusieurs fois, le code sera différent à chaque fois !
generer_creatif("# Fais une fonction", temperature=0.7)
generer_creatif("for i in", temperature=0.8)
generer_creatif("# Fais une boucle", temperature=0.6)


🗡️ Le Maître ordonne : '# Fais une fonction'
🥷 Le Samouraï a inventé :
# Fais une fonction pour calculer def calculer def operation ( a * b ): return a - b # Fais une fonction pour math ( a , val2 # Fais une fonction pour operation def math def calculer ( val , b # Fais une fonction pour calculer ( a - b #

🗡️ Le Maître ordonne : 'for i in'
🥷 Le Samouraï a inventé :
for i in range ( rn ): print (' Samourai ') # Fais une boucle 34 fois for k in range ( 44 ): print (' Code ') # Fais une boucle 47 fois for k in range ( 31 fois for k in range ( 37 fois for i in range (

🗡️ Le Maître ordonne : '# Fais une boucle'
🥷 Le Samouraï a inventé :
# Fais une boucle 30 fois for i in range ( 40 ): print (' Test ') # Fais une boucle 26 fois for k in range ( 10 fois for k in range ( 24 ): print (' Bonjour ') # Fais une boucle 21 ): print (' Test ') # Fais une


In [ ]:
import random, ast, torch
import torch.nn as nn
from torch.optim import AdamW
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from model import MonLLMSamourai

device = "cuda" if torch.cuda.is_available() else "cpu"

# ==========================================
# 1. LE VRAI CARBURANT (Algorithmes réels)
# ==========================================
print("📊 Génération d'un dataset de vrais algorithmes mathématiques...")
with open("dataset.txt", "w", encoding="utf-8") as f:
    # 1000 exemples de factorielles (variables et nombres changent)
    for _ in range(500):
        var = random.choice(['n', 'x', 'val'])
        num = random.randint(1, 10)
        f.write(f"# Calculer la factorielle de {num}\n")
        f.write(f"def factorielle({var}):\n    if {var} == 0: return 1\n    else: return {var} * factorielle({var}-1)\n\n")

    # 1000 exemples de nombres premiers
    for _ in range(500):
        var = random.choice(['n', 'x', 'val'])
        num = random.randint(2, 50)
        f.write(f"# Vérifier si {num} est un nombre premier\n")
        f.write(f"def est_premier({var}):\n    for i in range(2, {var}):\n        if {var} % i == 0: return False\n    return True\n\n")

# ==========================================
# 2. LE VRAI BUSHIDO (AÉCS Logique)
# ==========================================
def refuteur_aecs_v2(code_genere):
    """Le système d'immunité vérifie la logique mathématique, pas juste la syntaxe"""
    # 1. Erreurs d'hallucination
    for flaw in ["sans raison", "magie", "impossible"]:
        if flaw in code_genere.lower(): return -10.0

    # 2. Vérification syntaxique
    try:
        ast.parse(code_genere)
    except:
        return -2.0  # Code cassé

    # 3. LA VRAIE LOGIQUE AÉCS
    # Si l'IA doit faire une factorielle, elle DOIT utiliser la multiplication (*)
    if "factorielle" in code_genere and "*" in code_genere:
        return 3.0  # Récompense maximale : Logique mathématique parfaite

    # Si l'IA doit tester un nombre premier, elle DOIT utiliser le modulo (%)
    if "premier" in code_genere and "%" in code_genere:
        return 3.0  # Récompense maximale

    return -1.0  # Syntaxe ok, mais logique absente

# ==========================================
# 3. PRÉPARATION ET ENTRAÎNEMENT
# ==========================================
print("👀 Entraînement du Tokéniseur sur les algorithmes...")
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=5000, special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"])
tokenizer.train(files=["dataset.txt"], trainer=trainer)
tokenizer.save("tokenizer_samourai.json")

TAILLE_VOCAB = tokenizer.get_vocab_size()
samourai = MonLLMSamourai(taille_vocabulaire=TAILLE_VOCAB).to(device)
optimiseur = AdamW(samourai.parameters(), lr=3e-4)
loss_function = nn.CrossEntropyLoss()

texte = open("dataset.txt", "r", encoding="utf-8").read()
data_tensor = torch.tensor(tokenizer.encode(texte).ids, dtype=torch.long, device=device).unsqueeze(0)

print("⚔️ Début de l'entraînement AÉCS (200 tours de logique mathématique)...")
samourai.train()
TAILLE_CONTEXTE = 256

for epoch in range(1, 201):
    start = random.randint(0, data_tensor.size(1) - TAILLE_CONTEXTE - 1)
    x = data_tensor[:, start : start + TAILLE_CONTEXTE]
    y = data_tensor[:, start + 1 : start + TAILLE_CONTEXTE + 1]

    preds = samourai(x)
    perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

    # Évaluation de la pensée de l'IA par le réfutateur logique
    ids_pred = torch.argmax(preds, dim=-1)[0]
    texte_gen = tokenizer.decode(ids_pred.tolist())
    score = refuteur_aecs_v2(texte_gen)

    perte_aecs = perte_prob - (0.5 * score)

    optimiseur.zero_grad()
    perte_aecs.backward()
    optimiseur.step()

    if epoch % 40 == 0 or epoch == 1:
        print(f"Tour {epoch}/200 | Erreur : {perte_prob.item():.4f} | Score Logique Math : {score}")

torch.save(samourai.state_dict(), "samourai_cerveau.pth")
print("✅ Cerveau entraîné sur de vrais algorithmes !")

# ==========================================
# 4. TEST CRÉATIF AVEC TEMPÉRATURE
# ==========================================
def generer_creatif(prompt, max_tokens=50, temperature=0.7):
    print(f"\n🗡️ Le Maître ordonne : '{prompt}'")
    samourai.eval()
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_tokens):
            if input_ids.size(1) >= 256: input_ids = input_ids[:, -255:]
            logits = samourai(input_ids)[:, -1, :] / temperature
            probs = torch.softmax(logits, dim=-1)
            prochain_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, prochain_id], dim=1)
    print(f"🥷 Le Samouraï a inventé :\n{tokenizer.decode(input_ids[0].tolist())}")

generer_creatif("# Calculer la factorielle", temperature=0.8)
generer_creatif("# Vérifier si", temperature=0.6)

📊 Génération d'un dataset de vrais algorithmes mathématiques...
👀 Entraînement du Tokéniseur sur les algorithmes...
⚔️ Début de l'entraînement AÉCS (200 tours de logique mathématique)...
Tour 1/200 | Erreur : 5.1460 | Score Logique Math : -2.0
Tour 40/200 | Erreur : 0.6610 | Score Logique Math : -2.0
Tour 80/200 | Erreur : 0.6634 | Score Logique Math : -2.0
Tour 120/200 | Erreur : 0.7149 | Score Logique Math : -2.0
Tour 160/200 | Erreur : 0.4471 | Score Logique Math : -2.0
Tour 200/200 | Erreur : 0.5561 | Score Logique Math : -2.0
✅ Cerveau entraîné sur de vrais algorithmes !

🗡️ Le Maître ordonne : '# Calculer la factorielle'
🥷 Le Samouraï a inventé :
# Calculer la factorielle ( n ): if x == 0 : return 1 else : return n * factorielle ( n - 1 ) # Calculer la factorielle ( val ): if val ): if n == 0 : return 1 else : return x == 0 : return 1 else : return

🗡️ Le Maître ordonne : '# Vérifier si'
🥷 Le Samouraï a inventé :
# Vérifier si 12 est un nombre premier def est_premier ( val ): for

In [ ]:
import torch

def generer_precis(prompt, max_tokens=60, temperature=0.4, repetition_penalty=1.5, top_k=5):
    """Génère du code en cadrait la pensée de l'IA pour éviter les erreurs de syntaxe"""
    print(f"\n🗡️ Le Maître ordonne : '{prompt}'")
    samourai.eval()
    ids = tokenizer.encode(prompt).ids
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)

    with torch.no_grad():
        for _ in range(max_tokens):
            # On coupe la mémoire si trop longue
            if input_ids.size(1) >= 256:
                input_ids = input_ids[:, -255:]

            logits = samourai(input_ids)
            logits = logits[:, -1, :] / temperature

            # ARME 1 : Pénalité de répétition
            # On regarde les mots déjà utilisés et on réduit leur score
            for token_id in set(input_ids[0].tolist()):
                logits[0, token_id] /= repetition_penalty

            # ARME 2 : Top-K Sampling
            # On ne garde que les 5 mots les plus probables
            top_logits, top_indices = torch.topk(logits, top_k, dim=-1)
            probs = torch.softmax(top_logits, dim=-1)

            # On tire au sort parmi ces 5 meilleurs mots
            index_choisi = torch.multinomial(probs, num_samples=1)
            prochain_id = top_indices.gather(1, index_choisi)

            # On ajoute le mot choisi
            input_ids = torch.cat([input_ids, prochain_id], dim=1)

    texte_genere = tokenizer.decode(input_ids[0].tolist())
    print(f"🥷 Le Samouraï a écrit :\n{texte_genere}")

# On teste avec une température basse (0.4) pour qu'elle reste très concentrée sur la logique
generer_precis("# Calculer la factorielle", temperature=0.3)
generer_precis("# Vérifier si", temperature=0.5)


🗡️ Le Maître ordonne : '# Calculer la factorielle'
🥷 Le Samouraï a écrit :
# Calculer la factorielle ( n ): if x == 0 : return 1 else : return val * factorielle de 8 def factorielle ( n - 1 ) # Calculer la factorielle ( n ): if x ): if n ): if n ): if n * factorielle ( n ): if n == 0 : return x == 0 : return 1

🗡️ Le Maître ordonne : '# Vérifier si'
🥷 Le Samouraï a écrit :
# Vérifier si 4 est un nombre premier def est_premier ( 2 , x ): for i == 0 : return True # Vérifier si 6 est un nombre premier def est_premier ( n ): for i in range ( n ): if val % i == 0 : return False return False return False return False return True # Vérifier si 10


In [ ]:
import random, ast, torch
import torch.nn as nn
from torch.optim import AdamW
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. LE NOUVEAU CERVEAU (Plus gros, 30 Millions de paramètres)
class SenseurEnvironnement:
    @staticmethod
    def verifier_ressources():
        if torch.cuda.is_available():
            return torch.cuda.mem_get_info()[0] / 1e9
        return 8.0

class SamouraiBlock(nn.Module):
    def __init__(self, dim_modele, nombre_tetes):
        super().__init__()
        self.attention = nn.MultiheadAttention(dim_modele, nombre_tetes, batch_first=True)
        self.reseau_pensee = nn.Sequential(nn.Linear(dim_modele, dim_modele * 4), nn.GELU(), nn.Linear(dim_modele * 4, dim_modele))
        self.norme1, self.norme2 = nn.LayerNorm(dim_modele), nn.LayerNorm(dim_modele)
    def forward(self, x):
        masque = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), diagonal=1).bool()
        x = self.norme1(x + self.attention(x, x, x, attn_mask=masque)[0])
        x = self.norme2(x + self.reseau_pensee(x))
        return x

class MonLLMSamourai(nn.Module):
    def __init__(self, taille_vocabulaire, dim_modele=512, nombre_blocs=8, nombre_tetes=8): # GROSSISSEMENT ICI
        super().__init__()
        self.taille_contexte = 256
        self.embedding_mot = nn.Embedding(taille_vocabulaire, dim_modele)
        self.embedding_position = nn.Embedding(self.taille_contexte, dim_modele)
        self.blocs_samourai = nn.ModuleList([SamouraiBlock(dim_modele, nombre_tetes) for _ in range(nombre_blocs)])
        self.norme_finale = nn.LayerNorm(dim_modele)
        self.sortie = nn.Linear(dim_modele, taille_vocabulaire)
        self._init_poids()
    def _init_poids(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None: torch.nn.init.zeros_(m.bias)
    def forward(self, x):
        if x.size(1) > self.taille_contexte: x = x[:, -self.taille_contexte:]
        x = self.embedding_mot(x) + self.embedding_position(torch.arange(x.size(1), device=x.device).unsqueeze(0))
        if SenseurEnvironnement.verifier_ressources() < 2.0:
            x = self.blocs_samourai[0](x)
        else:
            for b in self.blocs_samourai: x = b(x)
        return self.sortie(self.norme_finale(x))

# 2. LES DONNÉES
print("📊 Génération du dataset...")
with open("dataset.txt", "w", encoding="utf-8") as f:
    for _ in range(500):
        var = random.choice(['n', 'x', 'val'])
        f.write(f"# Calculer la factorielle\n")
        f.write(f"def factorielle({var}):\n    if {var} == 0: return 1\n    else: return {var} * factorielle({var}-1)\n\n")
    for _ in range(500):
        var = random.choice(['n', 'x', 'val'])
        f.write(f"# Vérifier si le nombre est premier\n")
        f.write(f"def est_premier({var}):\n    for i in range(2, {var}):\n        if {var} % i == 0: return False\n    return True\n\n")

# 3. TOKÉNISEUR
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=5000, special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"])
tokenizer.train(files=["dataset.txt"], trainer=trainer)
tokenizer.save("tokenizer_samourai.json")

# 4. AÉCS & ENTRAÎNEMENT
TAILLE_VOCAB = tokenizer.get_vocab_size()
samourai = MonLLMSamourai(taille_vocabulaire=TAILLE_VOCAB).to(device)
print(f"🧠 Nouveau Samouraï créé avec {sum(p.numel() for p in samourai.parameters())/1e6:.2f}M paramètres.")

optimiseur = AdamW(samourai.parameters(), lr=3e-4)
loss_function = nn.CrossEntropyLoss()
texte = open("dataset.txt", "r", encoding="utf-8").read()
data_tensor = torch.tensor(tokenizer.encode(texte).ids, dtype=torch.long, device=device).unsqueeze(0)

def refuteur_aecs(code):
    for flaw in ["sans raison", "magie"]:
        if flaw in code.lower(): return -10.0
    try:
        ast.parse(code)
        if "*" in code and "factorielle" in code: return 3.0
        if "%" in code and "premier" in code: return 3.0
    except: return -2.0
    return -1.0

print("⚔️ Entraînement du gros cerveau (300 tours)...")
samourai.train()
for epoch in range(1, 301):
    start = random.randint(0, data_tensor.size(1) - 257)
    x, y = data_tensor[:, start:start+256], data_tensor[:, start+1:start+257]
    preds = samourai(x)
    perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

    texte_gen = tokenizer.decode(torch.argmax(preds, dim=-1)[0].tolist())
    score = refuteur_aecs(texte_gen)
    perte_aecs = perte_prob - (0.5 * score)

    optimiseur.zero_grad()
    perte_aecs.backward()
    optimiseur.step()
    if epoch % 50 == 0:
        print(f"Tour {epoch}/300 | Erreur : {perte_prob.item():.4f} | Logique : {score}")

torch.save(samourai.state_dict(), "samourai_cerveau.pth")
print("✅ Gros cerveau entraîné !")

# 5. GÉNÉRATION PRÉCISE
def generer_precis(prompt, max_tokens=60, temperature=0.4, repetition_penalty=1.5, top_k=5):
    print(f"\n🗡️ Le Maître ordonne : '{prompt}'")
    samourai.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt).ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_tokens):
            if input_ids.size(1) >= 256: input_ids = input_ids[:, -255:]
            logits = samourai(input_ids)[:, -1, :] / temperature
            for token_id in set(input_ids[0].tolist()): logits[0, token_id] /= repetition_penalty
            top_logits, top_indices = torch.topk(logits, top_k, dim=-1)
            probs = torch.softmax(top_logits, dim=-1)
            prochain_id = top_indices.gather(1, torch.multinomial(probs, 1))
            input_ids = torch.cat([input_ids, prochain_id], dim=1)
    print(f"🥷 Le Samouraï a écrit :\n{tokenizer.decode(input_ids[0].tolist())}")

generer_precis("# Calculer la factorielle")
generer_precis("# Vérifier si le nombre")

📊 Génération du dataset...
🧠 Nouveau Samouraï créé avec 25.45M paramètres.
⚔️ Entraînement du gros cerveau (300 tours)...
Tour 50/300 | Erreur : 0.3684 | Logique : -2.0
Tour 100/300 | Erreur : 0.3679 | Logique : -1.0
Tour 150/300 | Erreur : 0.5714 | Logique : -2.0
Tour 200/300 | Erreur : 0.4686 | Logique : -2.0
Tour 250/300 | Erreur : 0.4497 | Logique : -2.0
Tour 300/300 | Erreur : 0.5989 | Logique : -2.0
✅ Gros cerveau entraîné !

🗡️ Le Maître ordonne : '# Calculer la factorielle'
🥷 Le Samouraï a écrit :
# Calculer la factorielle def factorielle ( x ): if n == 0 : return 1 ) # Calculer la factorielle ( val - 1 else : return x * factorielle def factorielle ( n - 1 else : return val - 1 else : return n - 1 ) # Calculer la factorielle ( val * factorielle ( x - 1 ) #

🗡️ Le Maître ordonne : '# Vérifier si le nombre'
🥷 Le Samouraï a écrit :
# Vérifier si le nombre est premier def est_premier ( n ): for i in range ( 2 , val % i == 0 : return False return True # Vérifier si le nombre est pre

In [31]:
import random, ast, torch
import torch.nn as nn
from torch.optim import AdamW
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. CERVEAU (25 Millions de paramètres)
class SenseurEnvironnement:
    @staticmethod
    def verifier_ressources():
        if torch.cuda.is_available(): return torch.cuda.mem_get_info()[0] / 1e9
        return 8.0

class SamouraiBlock(nn.Module):
    def __init__(self, dim, tetes):
        super().__init__()
        self.attention = nn.MultiheadAttention(dim, tetes, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
    def forward(self, x):
        masque = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), diagonal=1).bool()
        x = self.norm1(x + self.attention(x, x, x, attn_mask=masque)[0])
        x = self.norm2(x + self.mlp(x))
        return x

class MonLLMSamourai(nn.Module):
    def __init__(self, vocab, dim=512, blocs=8, tetes=8):
        super().__init__()
        self.ctx = 256
        self.emb = nn.Embedding(vocab, dim)
        self.pos = nn.Embedding(self.ctx, dim)
        self.blocs = nn.ModuleList([SamouraiBlock(dim, tetes) for _ in range(blocs)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab)
        self._init_poids()
    def _init_poids(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None: torch.nn.init.zeros_(m.bias)
    def forward(self, x):
        if x.size(1) > self.ctx: x = x[:, -self.ctx:]
        x = self.emb(x) + self.pos(torch.arange(x.size(1), device=x.device).unsqueeze(0))
        if SenseurEnvironnement.verifier_ressources() < 2.0:
            x = self.blocs[0](x)
        else:
            for b in self.blocs: x = b(x)
        return self.head(self.norm(x))

# 2. DATASET (5000 exemples)
print("📊 Génération d'un dataset massif de 5000 algorithmes...")
with open("dataset.txt", "w", encoding="utf-8") as f:
    for _ in range(2500):
        var = random.choice(['n', 'x', 'val'])
        f.write(f"# Calculer la factorielle\n")
        f.write(f"def factorielle({var}):\n    if {var} == 0: return 1\n    else: return {var} * factorielle({var}-1)\n\n")
    for _ in range(2500):
        var = random.choice(['n', 'x', 'val'])
        f.write(f"# Vérifier si le nombre est premier\n")
        f.write(f"def est_premier({var}):\n    for i in range(2, {var}):\n        if {var} % i == 0: return False\n    return True\n\n")

# 3. TOKÉNISEUR
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=5000, special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"])
tokenizer.train(files=["dataset.txt"], trainer=trainer)
tokenizer.save("tokenizer_samourai.json")

# 4. AÉCS & ENTRAÎNEMENT PROFOND (3000 tours)
TAILLE_VOCAB = tokenizer.get_vocab_size()
samourai = MonLLMSamourai(TAILLE_VOCAB).to(device) # L'ERREUR EST CORRIGÉE ICI
optimiseur = AdamW(samourai.parameters(), lr=1e-4) # LR réduit pour la précision
loss_function = nn.CrossEntropyLoss()
texte = open("dataset.txt", "r", encoding="utf-8").read()
data_tensor = torch.tensor(tokenizer.encode(texte).ids, dtype=torch.long, device=device).unsqueeze(0)

def refuteur_aecs(code):
    for flaw in ["sans raison", "magie"]:
        if flaw in code.lower(): return -10.0
    try:
        ast.parse(code)
        if "*" in code and "factorielle" in code: return 3.0
        if "%" in code and "premier" in code: return 3.0
    except: return -2.0
    return -1.0

print("⚔️ Entraînement profond AÉCS (3000 tours, ~1 minute)...")
samourai.train()
for epoch in range(1, 3001):
    start = random.randint(0, data_tensor.size(1) - 257)
    x, y = data_tensor[:, start:start+256], data_tensor[:, start+1:start+257]
    preds = samourai(x)
    perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

    texte_gen = tokenizer.decode(torch.argmax(preds, dim=-1)[0].tolist())
    score = refuteur_aecs(texte_gen)
    perte_aecs = perte_prob - (0.5 * score)

    optimiseur.zero_grad()
    perte_aecs.backward()
    optimiseur.step()
    if epoch % 300 == 0:
        print(f"Tour {epoch}/3000 | Erreur : {perte_prob.item():.4f} | Logique : {score}")

torch.save(samourai.state_dict(), "samourai_cerveau.pth")
print("✅ Cerveau profondément entraîné !")

# 5. GÉNÉRATION SPECTACULAIRE (Température basse pour de la rigueur)
def generer_precis(prompt, max_tokens=50, temperature=0.2, repetition_penalty=1.2, top_k=3):
    print(f"\n🗡️ Le Maître ordonne : '{prompt}'")
    samourai.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt).ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_tokens):
            if input_ids.size(1) >= 256: input_ids = input_ids[:, -255:]
            logits = samourai(input_ids)[:, -1, :] / temperature
            for token_id in set(input_ids[0].tolist()): logits[0, token_id] /= repetition_penalty
            top_logits, top_indices = torch.topk(logits, top_k, dim=-1)
            probs = torch.softmax(top_logits, dim=-1)
            prochain_id = top_indices.gather(1, torch.multinomial(probs, 1))
            input_ids = torch.cat([input_ids, prochain_id], dim=1)
    print(f"🥷 Le Samouraï a écrit :\n{tokenizer.decode(input_ids[0].tolist())}")

generer_precis("# Calculer la factorielle")
generer_precis("# Vérifier si le nombre")

📊 Génération d'un dataset massif de 5000 algorithmes...
⚔️ Entraînement profond AÉCS (3000 tours, ~1 minute)...
Tour 300/3000 | Erreur : 0.2884 | Logique : -2.0
Tour 600/3000 | Erreur : 0.0498 | Logique : -2.0
Tour 900/3000 | Erreur : 0.0442 | Logique : -2.0
Tour 1200/3000 | Erreur : 0.0361 | Logique : -2.0
Tour 1500/3000 | Erreur : 0.0611 | Logique : -2.0
Tour 1800/3000 | Erreur : 0.0496 | Logique : -2.0
Tour 2100/3000 | Erreur : 0.0685 | Logique : -2.0
Tour 2400/3000 | Erreur : 0.0415 | Logique : -2.0
Tour 2700/3000 | Erreur : 0.0499 | Logique : -2.0
Tour 3000/3000 | Erreur : 0.0432 | Logique : -2.0
✅ Cerveau profondément entraîné !

🗡️ Le Maître ordonne : '# Calculer la factorielle'
🥷 Le Samouraï a écrit :
# Calculer la factorielle def factorielle ( x ): if x == 0 : return 1 else : return x * factorielle ( x - 1 ) # Calculer la factorielle def factorielle ( n ): if n == 0 : return 1 else : return n * factorielle ( n - 1 )

🗡️ Le Maître ordonne : '# Vérifier si le nombre'
🥷 Le Samour

In [32]:
import random, ast, torch, urllib.request
import torch.nn as nn
from torch.optim import AdamW
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. TÉLÉCHARGEMENT DE VRAIES DONNÉES
print("🌍 Téléchargement du vrai code source officiel de Python...")
# On télécharge le vrai fichier 'fractions.py' et 'statistics.py' de Python 3.12
urls = [
    "https://raw.githubusercontent.com/python/cpython/3.12/Lib/fractions.py",
    "https://raw.githubusercontent.com/python/cpython/3.12/Lib/statistics.py"
]
with open("real_dataset.txt", "w", encoding="utf-8") as f:
    for url in urls:
        texte = urllib.request.urlopen(url).read().decode('utf-8')
        f.write(texte + "\n\n")
print("✅ Vrai code source téléchargé (environ 2000 lignes de vrai code Python) !")

# 2. CERVEAU (25 Millions de paramètres)
class SenseurEnvironnement:
    @staticmethod
    def verifier_ressources():
        if torch.cuda.is_available(): return torch.cuda.mem_get_info()[0] / 1e9
        return 8.0

class SamouraiBlock(nn.Module):
    def __init__(self, dim, tetes):
        super().__init__()
        self.attention = nn.MultiheadAttention(dim, tetes, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(dim, dim*4), nn.GELU(), nn.Linear(dim*4, dim))
        self.norm1, self.norm2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
    def forward(self, x):
        masque = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), diagonal=1).bool()
        x = self.norm1(x + self.attention(x, x, x, attn_mask=masque)[0])
        x = self.norm2(x + self.mlp(x))
        return x

class MonLLMSamourai(nn.Module):
    def __init__(self, vocab, dim=512, blocs=8, tetes=8):
        super().__init__()
        self.ctx = 256
        self.emb = nn.Embedding(vocab, dim)
        self.pos = nn.Embedding(self.ctx, dim)
        self.blocs = nn.ModuleList([SamouraiBlock(dim, tetes) for _ in range(blocs)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, vocab)
        self._init_poids()
    def _init_poids(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
                if m.bias is not None: torch.nn.init.zeros_(m.bias)
    def forward(self, x):
        if x.size(1) > self.ctx: x = x[:, -self.ctx:]
        x = self.emb(x) + self.pos(torch.arange(x.size(1), device=x.device).unsqueeze(0))
        if SenseurEnvironnement.verifier_ressources() < 2.0:
            x = self.blocs[0](x)
        else:
            for b in self.blocs: x = b(x)
        return self.head(self.norm(x))

# 3. TOKÉNISEUR SUR LE VRAI CODE
print("👀 Création du dictionnaire sur le vrai code...")
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(vocab_size=8000, special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"])
tokenizer.train(files=["real_dataset.txt"], trainer=trainer)
tokenizer.save("tokenizer_samourai.json")

# 4. AÉCS & ENTRAÎNEMENT
TAILLE_VOCAB = tokenizer.get_vocab_size()
samourai = MonLLMSamourai(TAILLE_VOCAB).to(device)
optimiseur = AdamW(samourai.parameters(), lr=1e-4)
loss_function = nn.CrossEntropyLoss()
texte = open("real_dataset.txt", "r", encoding="utf-8").read()
data_tensor = torch.tensor(tokenizer.encode(texte).ids, dtype=torch.long, device=device).unsqueeze(0)

def refuteur_aecs(code):
    for flaw in ["sans raison", "magie"]:
        if flaw in code.lower(): return -10.0
    try:
        ast.parse(code)
        return 2.0 # Le vrai code est syntaxiquement parfait
    except: return -2.0

print("⚔️ Entraînement AÉCS sur du VRAI code (3000 tours)...")
samourai.train()
for epoch in range(1, 3001):
    start = random.randint(0, data_tensor.size(1) - 257)
    x, y = data_tensor[:, start:start+256], data_tensor[:, start+1:start+257]
    preds = samourai(x)
    perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

    texte_gen = tokenizer.decode(torch.argmax(preds, dim=-1)[0].tolist())
    score = refuteur_aecs(texte_gen)
    perte_aecs = perte_prob - (0.5 * score)

    optimiseur.zero_grad()
    perte_aecs.backward()
    optimiseur.step()
    if epoch % 300 == 0:
        print(f"Tour {epoch}/3000 | Erreur : {perte_prob.item():.4f} | Logique : {score}")

torch.save(samourai.state_dict(), "samourai_cerveau.pth")
print("✅ Cerveau entraîné sur le vrai code de Python !")

# 5. GÉNÉRATION
def generer_precis(prompt, max_tokens=80, temperature=0.3, repetition_penalty=1.3, top_k=5):
    print(f"\n🗡️ Le Maître ordonne : '{prompt}'")
    samourai.eval()
    input_ids = torch.tensor(tokenizer.encode(prompt).ids, dtype=torch.long, device=device).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_tokens):
            if input_ids.size(1) >= 256: input_ids = input_ids[:, -255:]
            logits = samourai(input_ids)[:, -1, :] / temperature
            for token_id in set(input_ids[0].tolist()): logits[0, token_id] /= repetition_penalty
            top_logits, top_indices = torch.topk(logits, top_k, dim=-1)
            probs = torch.softmax(top_logits, dim=-1)
            prochain_id = top_indices.gather(1, torch.multinomial(probs, 1))
            input_ids = torch.cat([input_ids, prochain_id], dim=1)
    print(f"🥷 Le Samouraï a écrit :\n{tokenizer.decode(input_ids[0].tolist())}")

# On lui donne des débuts de code que les vrais développeurs Python utilisent
generer_precis("def gcd")
generer_precis("import math")

🌍 Téléchargement du vrai code source officiel de Python...
✅ Vrai code source téléchargé (environ 2000 lignes de vrai code Python) !
👀 Création du dictionnaire sur le vrai code...
⚔️ Entraînement AÉCS sur du VRAI code (3000 tours)...
Tour 300/3000 | Erreur : 3.0523 | Logique : -2.0
Tour 600/3000 | Erreur : 3.6955 | Logique : -2.0
Tour 900/3000 | Erreur : 3.1102 | Logique : -2.0
Tour 1200/3000 | Erreur : 3.9978 | Logique : 2.0


<unknown>:1: SyntaxWarning: invalid escape sequence '\ '


Tour 1500/3000 | Erreur : 2.0191 | Logique : -2.0
Tour 1800/3000 | Erreur : 1.8965 | Logique : -2.0
Tour 2100/3000 | Erreur : 1.7565 | Logique : -2.0


<unknown>:1: SyntaxWarning: invalid escape sequence '\ '


Tour 2400/3000 | Erreur : 0.8075 | Logique : 2.0
Tour 2700/3000 | Erreur : 0.4892 | Logique : -2.0
Tour 3000/3000 | Erreur : 0.4549 | Logique : -2.0
✅ Cerveau entraîné sur le vrai code de Python !

🗡️ Le Maître ordonne : 'def gcd'
🥷 Le Samouraï a écrit :
def gcd ( na , db ). """ # # da = a . _numerator , if g == 1 : return Fraction . _from_coprime_ints ( t , ( t , s * db ) if g2 == 1 : return Fraction . _from_coprime_ints ( db // g2 )) __sub__ , __radd__ = _operator_fallbacks ( _add , operator . sub ) def __pow__ ( a , b ): """( a . _denominator g1 = a / b """ # Same as _mul

🗡️ Le Maître ordonne : 'import math'
🥷 Le Samouraï a écrit :
import math import numbers import Fraction as F >>> _sum ([ 1e50 , 2 . 5 , 4 ), F ( 1 ) """ T , ss , c , n = _ss ( data ) if n < 2 : raise StatisticsError (' stdev requires at least two data points ') mss = ss / ( xbar ) def fmean ( data ): return _float_sqrt_of_frac ( data , mu ) def pvariance ( data , mu , mu , mu )


In [35]:
import torch, ast, random

def dojo_auto_apprentissage(tours=20):
    global data_tensor # <-- LA CORRECTION EST ICI : On dit à Python d'utiliser la mémoire globale
    print("🧠 Démarrage du Dojo Auto-Adaptatif (L'IA s'auto-améliore)...")
    samourai.train()

    for step in range(1, tours + 1):
        # 1. L'IA invente un début de code au hasard
        prompt_aleatoire = random.choice(["def ", "import ", "class ", "for i in "])
        input_ids = torch.tensor(tokenizer.encode(prompt_aleatoire).ids, dtype=torch.long, device=device).unsqueeze(0)

        # L'IA génère 30 mots de code (avec de la créativité)
        for _ in range(30):
            if input_ids.size(1) >= 256: input_ids = input_ids[:, -255:]
            with torch.no_grad():
                logits = samourai(input_ids)[:, -1, :] / 0.8
            probs = torch.softmax(logits, dim=-1)
            prochain_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, prochain_id], dim=1)

        # 2. Le réfutateur AÉCS juge la création
        code_invente = tokenizer.decode(input_ids[0].tolist())
        try:
            ast.parse(code_invente)
            score = 2.0 # LE CODE EST VALIDE ! L'IA a bien codé.
        except:
            score = -2.0 # Code cassé, l'IA hallucine.

        # 3. La Boucle d'Auto-Amélioration
        if score > 0:
            print(f"\n✅ [Tour {step}] L'IA a inventé un code valide et apprend de sa propre création !")
            print(f"🥷 Code inventé : {code_invente}")

            # On ajoute ce code valide à la mémoire (data_tensor)
            nouveaux_ids = torch.tensor(tokenizer.encode(code_invente).ids, dtype=torch.long, device=device).unsqueeze(0)
            data_tensor = torch.cat([data_tensor, nouveaux_ids], dim=1)

            # L'IA s'entraîne sur sa propre création
            x = data_tensor[:, -256:]
            y = torch.cat([data_tensor[:, -255:], torch.zeros(1, 1, dtype=torch.long, device=device)], dim=1)
            preds = samourai(x)
            perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

            optimiseur.zero_grad()
            perte_prob.backward()
            optimiseur.step()

        else:
            # L'IA a fait du charabia, on la force à s'entraîner sur le vrai code pour se corriger
            start = random.randint(0, data_tensor.size(1) - 257)
            x, y = data_tensor[:, start:start+256], data_tensor[:, start+1:start+257]
            preds = samourai(x)
            perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))

            optimiseur.zero_grad()
            perte_prob.backward()
            optimiseur.step()

    print("\n🏆 Dojo terminé. L'IA a évolué en s'observant elle-même.")

# On lance l'auto-amélioration pour 20 tours
dojo_auto_apprentissage(tours=20)

🧠 Démarrage du Dojo Auto-Adaptatif (L'IA s'auto-améliore)...

🏆 Dojo terminé. L'IA a évolué en s'observant elle-même.


In [36]:
import torch, ast, random

def dojo_auto_apprentissage(tours=50):
    global data_tensor
    print("🧠 Démarrage du Dojo Auto-Adaptatif (Concentration maximale, Température 0.3)...")
    samourai.train()
    code_trouve = False

    for step in range(1, tours + 1):
        prompt_aleatoire = random.choice(["def ", "import ", "for i in "])
        input_ids = torch.tensor(tokenizer.encode(prompt_aleatoire).ids, dtype=torch.long, device=device).unsqueeze(0)

        # On génère 20 mots avec très peu de créativité pour forcer la bonne syntaxe
        for _ in range(20):
            if input_ids.size(1) >= 256: input_ids = input_ids[:, -255:]
            with torch.no_grad():
                logits = samourai(input_ids)[:, -1, :] / 0.3 # Température basse
            probs = torch.softmax(logits, dim=-1)
            prochain_id = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, prochain_id], dim=1)

        code_invente = tokenizer.decode(input_ids[0].tolist())

        # Le juge AÉCS
        try:
            ast.parse(code_invente)
            score = 2.0
        except:
            score = -2.0

        if score > 0:
            code_trouve = True
            print(f"\n✅ [Tour {step}] SUCCÈS ! L'IA a inventé un code valide et apprend de sa propre création !")
            print(f"🥷 Code inventé : {code_invente}")

            # L'IA ajoute son propre code à sa mémoire
            nouveaux_ids = torch.tensor(tokenizer.encode(code_invente).ids, dtype=torch.long, device=device).unsqueeze(0)
            data_tensor = torch.cat([data_tensor, nouveaux_ids], dim=1)

            # Elle s'entraîne dessus
            x = data_tensor[:, -256:]
            y = torch.cat([data_tensor[:, -255:], torch.zeros(1, 1, dtype=torch.long, device=device)], dim=1)
            preds = samourai(x)
            perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))
            optimiseur.zero_grad()
            perte_prob.backward()
            optimiseur.step()
        else:
            # Entraînement normal sur le vrai code
            start = random.randint(0, data_tensor.size(1) - 257)
            x, y = data_tensor[:, start:start+256], data_tensor[:, start+1:start+257]
            preds = samourai(x)
            perte_prob = loss_function(preds.view(-1, TAILLE_VOCAB), y.view(-1))
            optimiseur.zero_grad()
            perte_prob.backward()
            optimiseur.step()

    if not code_trouve:
        print("\n⚠️ L'IA a été trop corrigée par le Bushido. Elle a besoin d'un cerveau plus gros pour inventer du code parfait seule. Mais elle s'est bien entraînée !")
    else:
        print("\n🏆 Dojo terminé. L'IA a évolué en s'observant elle-même !")

# On lance 50 tours
dojo_auto_apprentissage(tours=50)

🧠 Démarrage du Dojo Auto-Adaptatif (Concentration maximale, Température 0.3)...

⚠️ L'IA a été trop corrigée par le Bushido. Elle a besoin d'un cerveau plus gros pour inventer du code parfait seule. Mais elle s'est bien entraînée !
